# Extract the Lyrics of Taylor Swift's Songs From the Genius API

I follow this project from: https://github.com/adashofdata/taylor_swift_data


This project is modified to extract more songs from non-album musics and delete all the dupucated songs. I chose the dulex or extentive version of the album to include all the tracks.

## Connect to Genius

In [9]:
from bs4 import BeautifulSoup
import re
import lyricsgenius
import requests
from pathlib import Path
import pandas as pd
import numpy as np

def clean_up(song_title):

    if "Ft" in song_title:
        before_ft_pattern = re.compile(".*(?=\(Ft)")
        song_title_before_ft = before_ft_pattern.search(song_title).group(0)
        clean_song_title = song_title_before_ft.strip()
        clean_song_title = clean_song_title.replace("/", "-")
    
    else:
        song_title_no_lyrics = song_title.replace("Lyrics", "")
        clean_song_title = song_title_no_lyrics.strip()
        clean_song_title = clean_song_title.replace("/", "-")
    
    return clean_song_title

def get_all_songs_from_album(artist, album_name):
    
    artist = artist.replace(" ", "-")
    album_name = album_name.replace(" ", "-")
    
    response = requests.get(f"https://genius.com/albums/{artist}/{album_name}")
    html_string = response.text
    document = BeautifulSoup(html_string, "html.parser")
    song_title_tags = document.find_all("h3", attrs={"class": "chart_row-content-title"})
    song_titles = [song_title.text for song_title in song_title_tags]
    
    clean_songs = []
    for song_title in song_titles:
        clean_song = clean_up(song_title)
        clean_songs.append(clean_song)
        
    return clean_songs

def download_album_lyrics(artist, album_name): 
    
    # You will need to go to Genius Developers to get your own client access token
    with open("access_token.txt") as f:
        client_access_token = f.read().strip()
    LyricsGenius = lyricsgenius.Genius(client_access_token)
    LyricsGenius.remove_section_headers = True
    
    clean_songs = get_all_songs_from_album(artist, album_name)
    
    for song in clean_songs:
        
        song_object = LyricsGenius.search_song(song, artist)
        
        if song_object != None:
            
            artist_title = artist.replace(" ", "-")
            album_title = album_name.replace(" ", "-")
            song_title = song.replace("/", "-")
            song_title = song.replace(" ", "-")
            
            custom_filename=f"{artist_title}_{album_title}/{song_title}"
            

            Path(f"{artist_title}_{album_title}").mkdir(parents=True, exist_ok=True)
            
            song_object.save_lyrics(filename=custom_filename, extension='txt', sanitize=False)
        
        else:
            print('No lyrics')

## Retrieve Taylor's song by album

In [6]:
### I ran a few lines at a time to not get a timeout error from Genius
#LyricsGenius = lyricsgenius.Genius('your token', timeout=15, retries=3)

download_album_lyrics("Taylor Swift", "Taylor Swift")
# download_album_lyrics("Taylor Swift", "Fearless-taylors-version")
# download_album_lyrics("Taylor Swift", "Speak-now-taylors-version")
# download_album_lyrics("Taylor Swift", "Red-taylors-version")
# download_album_lyrics("Taylor Swift", "1989-taylors-version")
# download_album_lyrics("Taylor Swift", "Reputation")
# download_album_lyrics("Taylor Swift", "Lover")
# download_album_lyrics("Taylor Swift", "Folklore-deluxe-version")
# download_album_lyrics("Taylor Swift", "Evermore-deluxe-version")
# download_album_lyrics("Taylor Swift", "Midnights-3am-edition")
# download_album_lyrics("Taylor Swift", "The-tortured-poets-department-the-anthology")
# download_album_lyrics("Taylor Swift", "The-life-of-a-showgirl")

Searching for "Tim McGraw" by Taylor Swift...
Done.


Taylor-Swift_Taylor-Swift/Tim-McGraw.txt already exists. Overwrite?
(y/n):  n


Skipping file save.

Searching for "Picture to Burn" by Taylor Swift...
Done.


Taylor-Swift_Taylor-Swift/Picture-to-Burn.txt already exists. Overwrite?
(y/n):  n


Skipping file save.

Searching for "Teardrops on My Guitar" by Taylor Swift...
Done.


...

Wrote Taylor-Swift_Taylor-Swift/Should've-Said-No.txt.
Searching for "Mary's Song (Oh My My My)" by Taylor Swift...
Done.


Taylor-Swift_Taylor-Swift/Mary's-Song-(Oh-My-My-My).txt already exists. Overwrite?
(y/n):  y


Wrote Taylor-Swift_Taylor-Swift/Mary's-Song-(Oh-My-My-My).txt.
Searching for "Our Song" by Taylor Swift...
Done.


Taylor-Swift_Taylor-Swift/Our-Song.txt already exists. Overwrite?
(y/n):  y


Wrote Taylor-Swift_Taylor-Swift/Our-Song.txt.


## Retrieve Taylor's non-albu songs

In [ ]:
# Some songs in a non-album, e.g., movie sound tracks and musics from the vault
def download_non_album_lyrics(artist, song_title, folder_name="non-album"):
    
    client_access_token = 'your token'
    LyricsGenius = lyricsgenius.Genius(client_access_token, timeout=15, retries=3)
    LyricsGenius.remove_section_headers = True
    
    song_object = LyricsGenius.search_song(song_title, artist)
    
    if song_object is not None:
        
        artist_title = artist.replace(" ", "-")
        clean_song_title = song_title.replace("/", "-").replace(" ", "-")
        
        custom_filename = f"{artist_title}_{folder_name}/{clean_song_title}"
        Path(f"{artist_title}_{folder_name}").mkdir(parents=True, exist_ok=True)
        song_object.save_lyrics(filename=custom_filename, extension='txt', sanitize=False)
        print(f"Downloaded: {song_title}")
        
    else:
        print(f"No lyrics found for: {song_title}")

download_non_album_lyrics("Taylor Swift", "Carolina")
download_non_album_lyrics("Taylor Swift", "All Of The Girls You Loved Before")
download_non_album_lyrics("Taylor Swift", "I Knew It, I Knew You")
download_non_album_lyrics("Taylor Swift", "You’re Losing Me (From The Vault)")
download_non_album_lyrics("Taylor Swift", "Only the Young")
download_non_album_lyrics("Taylor Swift", "Christmas Tree Farm")
download_non_album_lyrics("Taylor Swift", "The Joker and the Queen")
download_non_album_lyrics("Taylor Swift", "Beautiful Ghosts")
download_non_album_lyrics("Taylor Swift", "I Don't Wanna Live Forever")
download_non_album_lyrics("Taylor Swift", "Crazier")
download_non_album_lyrics("Taylor Swift", "Safe & Sound")
download_non_album_lyrics("Taylor Swift", "Eyes Open")

## Load Text Data into a Clean DataFrame

In [11]:
import re
import string

def clean_text(text):
    
    # Some light data cleaning - you will need to adjust based on your data
    text = text.replace('See Taylor Swift LiveGet tickets as low as $270', ' ') # remove ad
    text = re.sub('\d*Embed', ' ', text) # remove ending text with number + Embed
    
    return text

In [13]:
# Specify the folder names with the lyric data from Genius
directory_paths = ['Taylor-Swift_Taylor-Swift/',
                   'Taylor-Swift_Fearless-taylors-version/',
                   'Taylor-Swift_Speak-Now-taylors-version/',
                   'Taylor-Swift_Red-taylors-version/',
                   'Taylor-Swift_1989-taylors-version/',
                   'Taylor-Swift_Reputation/',
                   'Taylor-Swift_Lover/',
                   'Taylor-Swift_folklore-deluxe-version/',
                   'Taylor-Swift_evermore-deluxe-version/',
                   'Taylor-Swift_Midnights-3am-edition/',
                   'Taylor-Swift_The-tortured-poets-department-the-anthology',
                    'Taylor-Swift_The-life-of-a-showgirl/',
                    'Taylor-Swift_non-album/']

In [15]:
pd.options.display.max_rows = 500
pd.set_option('display.max_colwidth', 0)

ts_lyrics = pd.DataFrame({"Album": [],
                          "Song Name": [],
                          "Lyrics": []})

idx = 0

for i, album in enumerate(directory_paths):
    
    album_name = album[13:-1].replace("-", " ")
        
    for song in Path(album).glob('*.txt'):
        
        song_name = str(song).replace("-", " ").split("/")[1][:-4]
                
        full_text = open(song, encoding="utf-8")
        lyrics_list = full_text.readlines() #read()
        lyrics_list = [l.replace("\n", "") for l in lyrics_list]
        lyrics = ' '.join(lyrics_list)
        lyrics = re.sub(r'([a-z])([A-Z])', r'\1 \2', lyrics)  # split camelCase
        lyrics = clean_text(lyrics)
        full_text.close()
        
        ts_lyrics.loc[idx] = [album_name, song_name, lyrics]
        idx += 1

In [17]:
ts_lyrics["Album"].unique()

array(['Taylor Swift', 'Fearless taylors version',
       'Speak Now taylors version', 'Red taylors version',
       '1989 taylors version', 'Reputation', 'Lover',
       'folklore deluxe version', 'evermore deluxe version',
       'Midnights 3am edition',
       'The tortured poets department the antholog',
       'The life of a showgirl', 'non album'], dtype=object)

In [19]:
## Found that the name for this track this broken
ts_lyrics.loc[ts_lyrics["Song Name"] == 'Mr', "Song Name"] = "Mr. Perfectly Fine"

In [21]:
ts_lyrics[ts_lyrics["Album"]=="Fearless taylors version"]

,Album,Song Name,Lyrics
11,Fearless taylors version,Bye Bye Baby (Taylor's Version) [From the Vault],"It wasn't just like a movie The rain didn't soak through my clothes, down to my skin I'm drivin' away and I, I guess you could say This is the last time I'll drive this way again Lost in the gray and I try to grab at the fray 'Cause I, I still love you, but I can't Bye, bye to everything I thought was on my side Bye, bye, baby I want you bad, but it's come down to nothing And all I have is your sympathy 'Cause you took me home but you just couldn't keep me Bye, bye, baby Bye, bye, baby The picture frame is empty On the dresser, vacant just like me I see your writing on the dash Then back to your hesitation I was so sure of everything Everything I thought we'd always have Guess I never doubted it Then the here and the now floods in Feels like I'm becoming a part of your past Bye, bye to everything I thought was on my side Bye, bye, baby I want you bad, but it's come down to nothing And all I have is your sympathy 'Cause you took me home but you just couldn't keep me Bye, bye, baby And there's so much that I can't touch You're all I want, but it's not enough this time And all the pages are just slipping through my hands And I'm so scared of how this ends Bye, bye to everything I thought was on my side Bye, bye, baby I want you bad, but it's come down to nothing And all I have is your sympathy 'Cause you took me home but you just couldn't keep me Bye, bye to everything I thought was on my side Bye, bye, baby I want you bad, but it's come down to nothing And all I have is your sympathy 'Cause you took me home but you just couldn't keep me Oh, you took me home, I thought you were gonna keep me Bye, bye, baby Bye, bye, baby"
12,Fearless taylors version,The Best Day (Taylor's Version),"I'm five years old, it's gettin' cold, I've got my big coat on I hear your laugh and look up smilin' at you, I run and run Past the pumpkin patch and the tractor rides, look, now the sky is gold I hug your legs and fall asleep on the way home I don't know why all the trees change in the fall But I know you're not scared of anything at all Don't know if Snow White's house is near or far away But I know I had the best day with you today I'm thirteen now and don't know how my friends could be so mean I come home cryin' and you hold me tight and grab the keys And we drive and drive until we found a town far enough away And we talk and window shop 'til I've forgotten all their names I don't know who I'm gonna talk to now at school But I know I'm laughin' on the car ride home with you Don't know how long it's gonna take to feel okay But I know I had the best day with you today I have an excellent father, his strength is making me stronger God smiles on my little brother, inside and out, he's better than I am I grew up in a pretty house and I had space to run And I had the best days with you There is a video I found from back when I was three You set up a paint set in the kitchen and you're talkin' to me It's the age of princesses and pirate ships and the seven dwarves Daddy's smart and you're the prettiest lady in the whole wide world Now I know why all the trees change in the fall I know you were on my side even when I was wrong And I love you for givin' me your eyes For staying back and watchin' me shine And I didn't know if you knew So I'm taking this chance to say That I had the best day with you today"
13,Fearless taylors version,Fifteen (Taylor's Version),"You take a deep breath and you walk through the doors It's the mornin' of your very first day You say hi to your friends you ain't seen in a while Try and stay out of everybody's way It's your freshman year and you're gonna be here For the next four years in this town Hopin' one of those senior boys will wink at you and say ""You know, I haven't seen you around before"" 'Cause when you're fifteen and somebody tells you they love you You're gonna believe them And when you're fifteen, feelin' like ther

### Remove the tracks that are duplicated

In [23]:
# Remove 8 extra rows of data including duplicate songs, deluxe songs and non-songs
# to match the same 147 songs that were on the Spotify API list

ts_lyrics.drop(22, axis=0, inplace=True) # Love Story remix
ts_lyrics.drop(26, axis=0, inplace=True) # Forever and always (piano duplicate version)
ts_lyrics.drop(60, axis=0, inplace=True) # Speak now prolonge
ts_lyrics.drop(73, axis=0, inplace=True) # State of grace acoustic 
ts_lyrics.drop(87, axis=0, inplace=True) # Red message from Taylor
ts_lyrics.drop(97, axis=0, inplace=True) # 1989prolonge
ts_lyrics.drop(125, axis=0, inplace=True) # REP prolonge
ts_lyrics.drop(151, axis=0, inplace=True) # Folklore foreword
ts_lyrics.reset_index(inplace=True)

## Export the Lyrics as a CSV File

In [25]:
ts_lyrics.to_csv("taylor_swift_genius_data.csv", index=False)